In [1]:
import pandas as pd

df = pd.read_csv("nifty50_2007_to_now.csv")
print(df.columns)


Index(['Price', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='object')


In [2]:
df.head()

,Price,Close,High,Low,Open,Volume
0,Ticker,^NSEI,^NSEI,^NSEI,^NSEI,^NSEI
1,Date,NaN,NaN,NaN,NaN,NaN
2,2007-09-17,4494.64990234375,4549.0498046875,4482.85009765625,4518.4501953125,0
3,2007-09-18,4546.2001953125,4551.7998046875,4481.5498046875,4494.10009765625,0
4,2007-09-19,4732.35009765625,4739.0,4550.25,4550.25,0


In [ ]:

df = df.rename(columns={df.columns[0]: "Date"})


df["Date"] = pd.to_datetime(df["Date"], errors="coerce")


df = df.dropna(subset=["Date"])


for col in df.columns:
    if col != "Date":
        df[col] = pd.to_numeric(df[col], errors="coerce")


df = df.ffill().bfill()

df = df.set_index("Date")

print(df.head())
print(df.dtypes)

# Regime splitting
df1 = df.loc[: "2010-12-31"]              # Pre-2011
df2 = df.loc["2011-01-01":"2019-12-31"]   # 2011–2019
df3 = df.loc["2020-01-01":"2021-12-31"]   # COVID period
df4 = df.loc["2022-01-01":]               # 2022 onward

# Print sizes
print("\nSizes of each segment:")
print("df1 (2007–2010):", len(df1))
print("df2 (2011–2019):", len(df2))
print("df3 (2020–2021):", len(df3))
print("df4 (2022–present):", len(df4))


                  Close         High          Low         Open  Volume
Date                                                                  
2007-09-17  4494.649902  4549.049805  4482.850098  4518.450195       0
2007-09-18  4546.200195  4551.799805  4481.549805  4494.100098       0
2007-09-19  4732.350098  4739.000000  4550.250000  4550.250000       0
2007-09-20  4747.549805  4760.850098  4721.149902  4734.850098       0
2007-09-21  4837.549805  4855.700195  4733.700195  4752.950195       0
Close     float64
High      float64
Low       float64
Open      float64
Volume      int64
dtype: object

Sizes of each segment:
df1 (2007–2010): 808
df2 (2011–2019): 2193
df3 (2020–2021): 498
df4 (2022–present): 963


C:\Users\ASUS\AppData\Local\Temp\ipykernel_42328\3758940358.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df["Date"], errors="coerce")


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# Dictionary to store all results
results = {
    "XGBoost": {},
    "CatBoost": {},
    "TCN": {},
    "ARIMA": {},
    "SARIMA": {},
    "BiLSTM": {},
    "ConvLSTM": {}
}

datasets = {
    "df1": df1,
    "df2": df2,
    "df3": df3,
    "df4": df4
}

def prepare_data(df):
    df = df.copy()

 

    
    y = df[["High", "Low"]]           
    X = df.drop(["High", "Low"], axis=1)

    return train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

def evaluate_model(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    non_zero_idx = y_true != 0
    mape = np.mean(np.abs((y_true[non_zero_idx] - y_pred[non_zero_idx]) / y_true[non_zero_idx])) * 100

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mae  = mean_absolute_error(y_true, y_pred)

    return {
        "MAPE": mape,
        "RMSE": rmse,
        "R2": r2,
        "MAE": mae
    }


In [11]:
from statsmodels.tsa.arima.model import ARIMA

for name, df in datasets.items():
    y = df["High"]  # choose High or Low separately

    train_size = int(len(y) * 0.8)
    train, test = y[:train_size], y[train_size:]

    model = ARIMA(train, order=(2,1,2))
    model_fit = model.fit()
    preds = model_fit.forecast(len(test))

    results["ARIMA"][name] = evaluate_model(test, preds)


d:\Anacondaa\envs\tfenv\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
d:\Anacondaa\envs\tfenv\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
d:\Anacondaa\envs\tfenv\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
d:\Anacondaa\envs\tfenv\lib\site-packages\statsmodels\tsa\statespace\sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parame

In [9]:
results["ARIMA"]

{'df1': {'MAPE': 10.545246377799621,
  'RMSE': 163.67499796650466,
  'R2': -2.3108132735427467,
  'MAE': 138.48302705491383},
 'df2': {'MAPE': 6.273673198732937,
  'RMSE': 105.37688196699078,
  'R2': -0.010481944895258888,
  'MAE': 85.51033689102611},
 'df3': {'MAPE': 2.7951623016531237,
  'RMSE': 57.830059938411516,
  'R2': -3.0894167596476496,
  'MAE': 50.64446880873578},
 'df4': {'MAPE': 15.45336017495487,
  'RMSE': 658.6615141525367,
  'R2': -2.48595968890244,
  'MAE': 556.2142402588826}}

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Bidirectional, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


def create_seq(X, y, seq_len=25):
    X_seq, y_seq = [], []
    for i in range(seq_len, len(X)):
        X_seq.append(X[i-seq_len:i])
        y_seq.append(y[i])
    return np.array(X_seq), np.array(y_seq)



def metrics(y_true, y_pred, label):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true,  y_pred)
    mape = np.mean(np.abs((y_true - y_pred)/y_true))*100

    print(f"\n===== {label} =====")
    print("MAE :",  round(mae,3))
    print("RMSE:",  round(rmse,3))
    print("R2  :",  round(r2,3))
    print("MAPE:", round(mape,2), "%")

    return mae, rmse, r2, mape


def build_bilstm(input_shape):
    model = Sequential([

        Bidirectional(LSTM(96, return_sequences=True), input_shape=input_shape),
        Dropout(0.25),

        Bidirectional(LSTM(64, return_sequences=False)),
        Dropout(0.25),

        Dense(32, activation='relu'),
        Dense(2)
    ])

    model.compile(optimizer='adam', loss='mse')
    return model


def run_BiLSTM(df, name):

    print(f"\n================ BiLSTM for {name} ================")

    X = df[['Open','High','Low','Close']].values
    y = df[['High','Low']].values

    seq_len = 25
    train_size = max(int(len(df)*0.85), seq_len+3)
    train_size = min(train_size, len(df)-2)

    X_train_raw = X[:train_size]
    X_test_raw  = X[train_size:]

    y_train_raw = y[:train_size]
    y_test_raw  = y[train_size:]

    
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_train_scaled = scaler_X.fit_transform(X_train_raw)
    X_test_scaled  = scaler_X.transform(X_test_raw)

    y_train_scaled = scaler_y.fit_transform(y_train_raw)
    y_test_scaled  = scaler_y.transform(y_test_raw)

    X_scaled = np.vstack([X_train_scaled, X_test_scaled])
    y_scaled = np.vstack([y_train_scaled, y_test_scaled])

    X_seq, y_seq = create_seq(X_scaled, y_scaled, seq_len)

    split = train_size - seq_len
    X_train, X_test = X_seq[:split], X_seq[split:]
    y_train, y_test = y_seq[:split], y_seq[split:]

    batch = 16 if len(X_train)>50 else 4

    model = build_bilstm((seq_len, X_train.shape[2]))

    es = EarlyStopping(patience=6, restore_best_weights=True)
    lr = ReduceLROnPlateau(factor=0.5, patience=3)

    model.fit(
        X_train, y_train,
        epochs=60,
        batch_size=batch,
        validation_data=(X_test, y_test) if len(X_test)>1 else None,
        callbacks=[es, lr],
        verbose=1
    )

    
    pred_scaled = model.predict(X_test)
    pred = scaler_y.inverse_transform(pred_scaled)
    y_true = scaler_y.inverse_transform(y_test)

    pred_high, pred_low = pred[:,0], pred[:,1]
    true_high, true_low = y_true[:,0], y_true[:,1]

    m1 = metrics(true_high, pred_high, f"BiLSTM {name} High")
    m2 = metrics(true_low,  pred_low,  f"BiLSTM {name} Low")

    return (m1, m2)


In [ ]:

# df1, df2, df3, df4 must already be loaded before running this cell.

print("\n\n====== RUNNING BiLSTM ON ALL DATASETS ======\n")

results_bilstm = {}

for df, name in zip([df1, df2, df3, df4], ["DF1", "DF2", "DF3", "DF4"]):
    print(f"\n\n----- Training BiLSTM for {name} -----")
    try:
        res = run_BiLSTM(df, name)
        results_bilstm[name] = {
            "High": {
                "MAE":  res[0][0],
                "RMSE": res[0][1],
                "R2":   res[0][2],
                "MAPE": res[0][3]
            },
            "Low": {
                "MAE":  res[1][0],
                "RMSE": res[1][1],
                "R2":   res[1][2],
                "MAPE": res[1][3]
            }
        }
    except Exception as e:
        print(f"Error in {name}: {e}")

print("\n\n============= FINAL BiLSTM RESULTS =============\n")
results_bilstm




====== RUNNING BiLSTM ON ALL DATASETS ======



----- Training BiLSTM for DF1 -----

================ BiLSTM for DF1 ================


d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/60
42/42 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - loss: 0.1768 - val_loss: 0.0379 - learning_rate: 0.0010
Epoch 2/60
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0608 - val_loss: 0.0287 - learning_rate: 0.0010
Epoch 3/60
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0488 - val_loss: 0.0426 - learning_rate: 0.0010
Epoch 4/60
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0438 - val_loss: 0.0183 - learning_rate: 0.0010
Epoch 5/60
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0347 - val_loss: 0.1012 - learning_rate: 0.0010
Epoch 6/60
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0359 - val_loss: 0.0399 - learning_rate: 0.0010
Epoch 7/60
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0306 - val_loss: 0.0221 - learning_rate: 0.0010
Epoch 8/60
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0308 - val_loss: 0.0159 - learning_rate: 5.0000e-04
Epoch 9/60
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0245 - val_loss: 0.0072 - learning_rate: 5.0000e-04
Epoch 10/60

d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


115/115 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - loss: 0.0555 - val_loss: 0.0789 - learning_rate: 0.0010
Epoch 2/60
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0183 - val_loss: 0.0622 - learning_rate: 0.0010
Epoch 3/60
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0150 - val_loss: 0.0229 - learning_rate: 0.0010
Epoch 4/60
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0143 - val_loss: 0.0381 - learning_rate: 0.0010
Epoch 5/60
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0126 - val_loss: 0.0184 - learning_rate: 0.0010
Epoch 6/60
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0109 - val_loss: 0.0139 - learning_rate: 0.0010
Epoch 7/60
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0103 - val_loss: 0.0131 - learning_rate: 0.0010
Epoch 8/60
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0099 - val_loss: 0.0208 - learning_rate: 0.0010
Epoch 9/60
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0094 - val_loss: 0.0127 - learning_rate: 0.0010
Epoch 10/60


d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


25/25 ━━━━━━━━━━━━━━━━━━━━ 4s 29ms/step - loss: 0.2118 - val_loss: 0.2502 - learning_rate: 0.0010
Epoch 2/60
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0607 - val_loss: 0.0644 - learning_rate: 0.0010
Epoch 3/60
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0378 - val_loss: 0.0604 - learning_rate: 0.0010
Epoch 4/60
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0287 - val_loss: 0.0386 - learning_rate: 0.0010
Epoch 5/60
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0284 - val_loss: 0.0289 - learning_rate: 0.0010
Epoch 6/60
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0284 - val_loss: 0.0210 - learning_rate: 0.0010
Epoch 7/60
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0241 - val_loss: 0.0277 - learning_rate: 0.0010
Epoch 8/60
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0202 - val_loss: 0.0228 - learning_rate: 0.0010
Epoch 9/60
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0205 - val_loss: 0.0176 - learning_rate: 0.0010
Epoch 10/60
25/25 ━━━━━━━━━━━━

d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


50/50 ━━━━━━━━━━━━━━━━━━━━ 5s 20ms/step - loss: 0.1034 - val_loss: 0.0583 - learning_rate: 0.0010
Epoch 2/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0332 - val_loss: 0.0277 - learning_rate: 0.0010
Epoch 3/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0245 - val_loss: 0.0260 - learning_rate: 0.0010
Epoch 4/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0247 - val_loss: 0.0177 - learning_rate: 0.0010
Epoch 5/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0215 - val_loss: 0.0264 - learning_rate: 0.0010
Epoch 6/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0192 - val_loss: 0.0496 - learning_rate: 0.0010
Epoch 7/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0196 - val_loss: 0.0072 - learning_rate: 0.0010
Epoch 8/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0167 - val_loss: 0.0092 - learning_rate: 0.0010
Epoch 9/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0159 - val_loss: 0.0068 - learning_rate: 0.0010
Epoch 10/60
50/50 ━━━━━━━━━━━━

{'DF1': {'High': {'MAE': 51.36989706070697,
   'RMSE': 60.86888675080471,
   'R2': 0.9593999437250789,
   'MAPE': 0.8892684764641359},
  'Low': {'MAE': 52.465968397797134,
   'RMSE': 69.8009292197764,
   'R2': 0.941671886734982,
   'MAPE': 0.8921922724741815}},
 'DF2': {'High': {'MAE': 89.06337564114742,
   'RMSE': 121.55832441337502,
   'R2': 0.9451557333584615,
   'MAPE': 0.7897616369703017},
  'Low': {'MAE': 102.95232950227964,
   'RMSE': 131.963773677115,
   'R2': 0.9380200940110395,
   'MAPE': 0.9259147660467781}},
 'DF3': {'High': {'MAE': 212.60408854166667,
   'RMSE': 278.2339848738755,
   'R2': 0.5404434646573008,
   'MAPE': 1.208256662333411},
  'Low': {'MAE': 242.05856770833333,
   'RMSE': 334.0554625670294,
   'R2': 0.4050722193811942,
   'MAPE': 1.398606911039179}},
 'DF4': {'High': {'MAE': 124.07956627155173,
   'RMSE': 164.52009899959862,
   'R2': 0.869923294524023,
   'MAPE': 0.49327740186796515},
  'Low': {'MAE': 135.75033674568965,
   'RMSE': 168.11792940166586,
   'R2

In [ ]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from tcn import TCN


def create_seq(X, y, seq_len=25):
    X_seq, y_seq = [], []
    for i in range(seq_len, len(X)):
        X_seq.append(X[i-seq_len:i])
        y_seq.append(y[i])
    return np.array(X_seq), np.array(y_seq)


def metrics(y_true, y_pred, label):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true,  y_pred)
    mape = np.mean(np.abs((y_true - y_pred)/y_true))*100

    print(f"\n===== {label} =====")
    print("MAE :",  round(mae,3))
    print("RMSE:",  round(rmse,3))
    print("R2  :",  round(r2,3))
    print("MAPE:", round(mape,2), "%")

    return mae, rmse, r2, mape


def build_tcn(input_shape):
    model = Sequential([

        TCN(nb_filters=64,
            kernel_size=3,
            dilations=[1, 2, 4, 8, 16],
            dropout_rate=0.25,
            return_sequences=False,
            activation='relu',
            input_shape=input_shape),

        Dense(32, activation='relu'),
        Dropout(0.25),
        Dense(2)
    ])

    model.compile(optimizer='adam', loss='mse')
    return model


def run_TCN(df, name):

    print(f"\n================ TCN for {name} ================")

    # Features and target
    X = df[['Open','High','Low','Close']].values
    y = df[['High','Low']].values

    seq_len = 25
    train_size = max(int(len(df)*0.85), seq_len+3)
    train_size = min(train_size, len(df)-2)

    X_train_raw = X[:train_size]
    X_test_raw  = X[train_size:]

    y_train_raw = y[:train_size]
    y_test_raw  = y[train_size:]

    
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_train_scaled = scaler_X.fit_transform(X_train_raw)
    X_test_scaled  = scaler_X.transform(X_test_raw)

    y_train_scaled = scaler_y.fit_transform(y_train_raw)
    y_test_scaled  = scaler_y.transform(y_test_raw)

    X_scaled = np.vstack([X_train_scaled, X_test_scaled])
    y_scaled = np.vstack([y_train_scaled, y_test_scaled])

    X_seq, y_seq = create_seq(X_scaled, y_scaled, seq_len)

    split = train_size - seq_len

    X_train, X_test = X_seq[:split], X_seq[split:]
    y_train, y_test = y_seq[:split], y_seq[split:]

    batch = 16 if len(X_train)>50 else 4

    model = build_tcn((seq_len, X_train.shape[2]))

    es = EarlyStopping(patience=6, restore_best_weights=True)
    lr = ReduceLROnPlateau(factor=0.5, patience=3)

    model.fit(
        X_train, y_train,
        epochs=60,
        batch_size=batch,
        validation_data=(X_test, y_test) if len(X_test)>1 else None,
        callbacks=[es, lr],
        verbose=1
    )

    
    pred_scaled = model.predict(X_test)
    pred = scaler_y.inverse_transform(pred_scaled)
    y_true = scaler_y.inverse_transform(y_test)

    pred_high, pred_low = pred[:,0], pred[:,1]
    true_high, true_low = y_true[:,0], y_true[:,1]

    m1 = metrics(true_high, pred_high, f"TCN {name} High")
    m2 = metrics(true_low,  pred_low,  f"TCN {name} Low")

    return (m1, m2)


In [15]:
results_tcn = {}

for df, name in zip([df1, df2, df3, df4], ["DF1", "DF2", "DF3", "DF4"]):
    print(f"\n\n----- Training TCN for {name} -----")
    try:
        res = run_TCN(df, name)
        results_tcn[name] = {
            "High": {
                "MAE":  res[0][0],
                "RMSE": res[0][1],
                "R2":   res[0][2],
                "MAPE": res[0][3]
            },
            "Low": {
                "MAE":  res[1][0],
                "RMSE": res[1][1],
                "R2":   res[1][2],
                "MAPE": res[1][3]
            }
        }
    except Exception as e:
        print(f"Error in {name}: {e}")

results_tcn




----- Training TCN for DF1 -----

================ TCN for DF1 ================


d:\Anacondaa\envs\tfenv\lib\site-packages\tcn\tcn.py:268: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super(TCN, self).__init__(**kwargs)



Epoch 1/60
42/42 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step - loss: 11.6021 - val_loss: 1.3216 - learning_rate: 0.0010
Epoch 2/60
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 1.0532 - val_loss: 0.8994 - learning_rate: 0.0010
Epoch 3/60
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.7779 - val_loss: 0.5668 - learning_rate: 0.0010
Epoch 4/60
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.6591 - val_loss: 0.2813 - learning_rate: 0.0010
Epoch 5/60
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.5221 - val_loss: 0.2973 - learning_rate: 0.0010
Epoch 6/60
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.5034 - val_loss: 0.1707 - learning_rate: 0.0010
Epoch 7/60
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.4328 - val_loss: 0.0938 - learning_rate: 0.0010
Epoch 8/60
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.3655 - val_loss: 0.1349 - learning_rate: 0.0010
Epoch 9/60
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.3812 - val_loss: 0.1725 - learning_rate: 0.0010
Epoch 10/60
42/42 ━━━━━━

d:\Anacondaa\envs\tfenv\lib\site-packages\tcn\tcn.py:268: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super(TCN, self).__init__(**kwargs)


115/115 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - loss: 5.6578 - val_loss: 0.2162 - learning_rate: 0.0010
Epoch 2/60
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.5575 - val_loss: 0.4696 - learning_rate: 0.0010
Epoch 3/60
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.3668 - val_loss: 0.5295 - learning_rate: 0.0010
Epoch 4/60
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.2702 - val_loss: 0.5883 - learning_rate: 0.0010
Epoch 5/60
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.2288 - val_loss: 0.3891 - learning_rate: 5.0000e-04
Epoch 6/60
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.2085 - val_loss: 0.3736 - learning_rate: 5.0000e-04
Epoch 7/60
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.2001 - val_loss: 0.3627 - learning_rate: 5.0000e-04
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step

===== TCN DF2 High =====
MAE : 805.003
RMSE: 827.842
R2  : -1.544
MAPE: 7.05 %

===== TCN DF2 Low =====
MAE : 809.52
RMSE: 867.64
R2  : -1.679
MAPE: 7.15 %


----- Training TCN for DF3 --

d:\Anacondaa\envs\tfenv\lib\site-packages\tcn\tcn.py:268: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super(TCN, self).__init__(**kwargs)


25/25 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - loss: 22.2516 - val_loss: 3.7214 - learning_rate: 0.0010
Epoch 2/60
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 1.6502 - val_loss: 1.4153 - learning_rate: 0.0010
Epoch 3/60
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 1.2452 - val_loss: 0.9915 - learning_rate: 0.0010
Epoch 4/60
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.9042 - val_loss: 1.3812 - learning_rate: 0.0010
Epoch 5/60
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.7265 - val_loss: 1.6904 - learning_rate: 0.0010
Epoch 6/60
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.7643 - val_loss: 0.3838 - learning_rate: 0.0010
Epoch 7/60
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.5659 - val_loss: 0.8706 - learning_rate: 0.0010
Epoch 8/60
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.6201 - val_loss: 0.0734 - learning_rate: 0.0010
Epoch 9/60
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.5858 - val_loss: 0.8102 - learning_rate: 0.0010
Epoch 10/60
25/25 ━━━━━━━━━━━━

d:\Anacondaa\envs\tfenv\lib\site-packages\tcn\tcn.py:268: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super(TCN, self).__init__(**kwargs)


50/50 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step - loss: 16.9372 - val_loss: 0.7976 - learning_rate: 0.0010
Epoch 2/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.9101 - val_loss: 0.8089 - learning_rate: 0.0010
Epoch 3/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.6801 - val_loss: 0.2229 - learning_rate: 0.0010
Epoch 4/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.5319 - val_loss: 0.9438 - learning_rate: 0.0010
Epoch 5/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.5036 - val_loss: 0.5070 - learning_rate: 0.0010
Epoch 6/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.4348 - val_loss: 0.2758 - learning_rate: 0.0010
Epoch 7/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.4066 - val_loss: 0.4666 - learning_rate: 5.0000e-04
Epoch 8/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.3920 - val_loss: 0.2630 - learning_rate: 5.0000e-04
Epoch 9/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.3522 - val_loss: 0.1764 - learning_rate: 5.0000e-04
Epoch 10/60
50/50 ━━━━━━━

{'DF1': {'High': {'MAE': 305.7254578637295,
   'RMSE': 327.02850012535487,
   'R2': -0.17194470950042273,
   'MAPE': 5.168889020038975},
  'Low': {'MAE': 196.32518730788934,
   'RMSE': 224.96097519537668,
   'R2': 0.39414333180200267,
   'MAPE': 3.371578780944816}},
 'DF2': {'High': {'MAE': 805.0032947853343,
   'RMSE': 827.8421909038652,
   'R2': -1.5436475584096612,
   'MAPE': 7.052503620771941},
  'Low': {'MAE': 809.5201723974164,
   'RMSE': 867.6398850292486,
   'R2': -1.6792946499655375,
   'MAPE': 7.14861398171234}},
 'DF3': {'High': {'MAE': 617.642265625,
   'RMSE': 665.5560658296523,
   'R2': -1.6295890758040028,
   'MAPE': 3.462763640411088},
  'Low': {'MAE': 554.4408333333333,
   'RMSE': 622.529145226301,
   'R2': -1.0660768475247164,
   'MAPE': 3.137753887883484}},
 'DF4': {'High': {'MAE': 828.1425511853448,
   'RMSE': 858.0094096689696,
   'R2': -2.5379012432313437,
   'MAPE': 3.282773308289333},
  'Low': {'MAE': 991.2887526939655,
   'RMSE': 1017.8481007069005,
   'R2': -3

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau



# ---------------- SEQUENCE CREATION ----------------
def create_seq(X, y, seq_len=25):
    X_seq, y_seq = [], []
    for i in range(seq_len, len(X)):
        X_seq.append(X[i-seq_len:i])
        y_seq.append(y[i])
    return np.array(X_seq), np.array(y_seq)



# ---------------- METRICS ----------------
def metrics(y_true, y_pred, label):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true,  y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    print(f"\n===== {label} =====")
    print("MAE :",  round(mae, 3))
    print("RMSE:",  round(rmse, 3))
    print("R2  :",  round(r2, 3))
    print("MAPE:", round(mape, 2), "%")

    return mae, rmse, r2, mape




def build_lstm(input_shape):

    model = Sequential([

        LSTM(96, return_sequences=True, input_shape=input_shape),
        Dropout(0.25),

        LSTM(64, return_sequences=False),
        Dropout(0.25),

        Dense(32, activation='relu'),
        Dense(2)
    ])

    model.compile(optimizer='adam', loss='mse')
    return model



def run_LSTM(df, name):

    print(f"\n================ LSTM for {name} ================")

    # Features and target
    X = df[['Open', 'High', 'Low', 'Close']].values
    y = df[['High', 'Low']].values

    seq_len = 25
    train_size = max(int(len(df) * 0.85), seq_len + 3)
    train_size = min(train_size, len(df) - 2)

    X_train_raw = X[:train_size]
    X_test_raw  = X[train_size:]

    y_train_raw = y[:train_size]
    y_test_raw  = y[train_size:]

    # Scaling
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_train_scaled = scaler_X.fit_transform(X_train_raw)
    X_test_scaled  = scaler_X.transform(X_test_raw)

    y_train_scaled = scaler_y.fit_transform(y_train_raw)
    y_test_scaled  = scaler_y.transform(y_test_raw)

    X_scaled = np.vstack([X_train_scaled, X_test_scaled])
    y_scaled = np.vstack([y_train_scaled, y_test_scaled])

    # Create sequences
    X_seq, y_seq = create_seq(X_scaled, y_scaled, seq_len)

    split = train_size - seq_len
    X_train, X_test = X_seq[:split], X_seq[split:]
    y_train, y_test = y_seq[:split], y_seq[split:]

    # Adaptive batch size
    batch_size = 16 if len(X_train) > 50 else 4

    # Build model
    model = build_lstm((seq_len, X_train.shape[2]))

    # Callbacks
    es = EarlyStopping(patience=6, restore_best_weights=True)
    lr = ReduceLROnPlateau(factor=0.5, patience=3)

    # Train
    model.fit(
        X_train, y_train,
        epochs=60,
        batch_size=batch_size,
        validation_data=(X_test, y_test) if len(X_test) > 1 else None,
        callbacks=[es, lr],
        verbose=1
    )

    # Prediction
    pred_scaled = model.predict(X_test)
    pred = scaler_y.inverse_transform(pred_scaled)
    y_true = scaler_y.inverse_transform(y_test)

    pred_high, pred_low = pred[:, 0], pred[:, 1]
    true_high, true_low = y_true[:, 0], y_true[:, 1]

    m1 = metrics(true_high, pred_high, f"LSTM {name} High")
    m2 = metrics(true_low,  pred_low,  f"LSTM {name} Low")

    return (m1, m2)





print("\n\n====== RUNNING LSTM ON ALL DATASETS ======\n")

results_lstm = {}

# df1, df2, df3, df4 must already be defined
for df, name in zip([df1, df2, df3, df4], ["DF1", "DF2", "DF3", "DF4"]):
    print(f"\n\n----- Training LSTM for {name} -----")
    try:
        res = run_LSTM(df, name)
        results_lstm[name] = {
            "High": {
                "MAE":  res[0][0],
                "RMSE": res[0][1],
                "R2":   res[0][2],
                "MAPE": res[0][3]
            },
            "Low": {
                "MAE":  res[1][0],
                "RMSE": res[1][1],
                "R2":   res[1][2],
                "MAPE": res[1][3]
            }
        }
    except Exception as e:
        print(f"Error in {name}: {e}")

print("\n\n============= FINAL LSTM RESULTS =============\n")
results_lstm




====== RUNNING LSTM ON ALL DATASETS ======



----- Training LSTM for DF1 -----

================ LSTM for DF1 ================
Epoch 1/60


d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


42/42 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - loss: 0.1696 - val_loss: 0.1159 - learning_rate: 0.0010
Epoch 2/60
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0540 - val_loss: 0.0239 - learning_rate: 0.0010
Epoch 3/60
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0525 - val_loss: 0.0124 - learning_rate: 0.0010
Epoch 4/60
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0469 - val_loss: 0.0111 - learning_rate: 0.0010
Epoch 5/60
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0467 - val_loss: 0.0433 - learning_rate: 0.0010
Epoch 6/60
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0401 - val_loss: 0.0209 - learning_rate: 0.0010
Epoch 7/60
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0390 - val_loss: 0.0349 - learning_rate: 0.0010
Epoch 8/60
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0391 - val_loss: 0.0108 - learning_rate: 5.0000e-04
Epoch 9/60
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0354 - val_loss: 0.0089 - learning_rate: 5.0000e-04
Epoch 10/60
42/42 ━━━━

d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


115/115 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - loss: 0.0647 - val_loss: 0.0466 - learning_rate: 0.0010
Epoch 2/60
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0187 - val_loss: 0.0241 - learning_rate: 0.0010
Epoch 3/60
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0168 - val_loss: 0.0314 - learning_rate: 0.0010
Epoch 4/60
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0147 - val_loss: 0.0176 - learning_rate: 0.0010
Epoch 5/60
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0140 - val_loss: 0.0182 - learning_rate: 0.0010
Epoch 6/60
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0148 - val_loss: 0.0333 - learning_rate: 0.0010
Epoch 7/60
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0135 - val_loss: 0.0250 - learning_rate: 0.0010
Epoch 8/60
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0107 - val_loss: 0.0264 - learning_rate: 5.0000e-04
Epoch 9/60
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0108 - val_loss: 0.0291 - learning_rate: 5.0000e-04
Epoch

d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - loss: 0.1966 - val_loss: 0.1472 - learning_rate: 0.0010
Epoch 2/60
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0540 - val_loss: 0.0902 - learning_rate: 0.0010
Epoch 3/60
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0462 - val_loss: 0.0191 - learning_rate: 0.0010
Epoch 4/60
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0335 - val_loss: 0.0206 - learning_rate: 0.0010
Epoch 5/60
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0321 - val_loss: 0.0310 - learning_rate: 0.0010
Epoch 6/60
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0323 - val_loss: 0.0187 - learning_rate: 0.0010
Epoch 7/60
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0304 - val_loss: 0.0417 - learning_rate: 0.0010
Epoch 8/60
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0333 - val_loss: 0.0185 - learning_rate: 0.0010
Epoch 9/60
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0259 - val_loss: 0.0222 - learning_rate: 0.0010
Epoch 10/60
25/25 ━━━━━━━━━━━━

d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


50/50 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - loss: 0.2151 - val_loss: 0.0393 - learning_rate: 0.0010
Epoch 2/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0360 - val_loss: 0.0155 - learning_rate: 0.0010
Epoch 3/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0328 - val_loss: 0.0098 - learning_rate: 0.0010
Epoch 4/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0306 - val_loss: 0.0098 - learning_rate: 0.0010
Epoch 5/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0240 - val_loss: 0.0095 - learning_rate: 0.0010
Epoch 6/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0256 - val_loss: 0.0156 - learning_rate: 0.0010
Epoch 7/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0235 - val_loss: 0.0152 - learning_rate: 0.0010
Epoch 8/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0258 - val_loss: 0.0113 - learning_rate: 0.0010
Epoch 9/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0199 - val_loss: 0.0068 - learning_rate: 5.0000e-04
Epoch 10/60
50/50 ━━━━━━━━

{'DF1': {'High': {'MAE': 57.66521916623975,
   'RMSE': 72.34166491568045,
   'R2': 0.9426527077063899,
   'MAPE': 0.9747463866913916},
  'Low': {'MAE': 70.74080270235656,
   'RMSE': 86.97763416956226,
   'R2': 0.9094328500483304,
   'MAPE': 1.2118731469679656}},
 'DF2': {'High': {'MAE': 179.81637656724925,
   'RMSE': 222.92686118391651,
   'R2': 0.8155466165601865,
   'MAPE': 1.5700595575257028},
  'Low': {'MAE': 213.23629250569908,
   'RMSE': 260.1211267072298,
   'R2': 0.7591799104387905,
   'MAPE': 1.8822407561091805}},
 'DF3': {'High': {'MAE': 218.9146875,
   'RMSE': 281.1694076065603,
   'R2': 0.530695491063899,
   'MAPE': 1.2400688972695535},
  'Low': {'MAE': 248.15971354166666,
   'RMSE': 317.2050355064192,
   'R2': 0.4635771869094941,
   'MAPE': 1.422859954138442}},
 'DF4': {'High': {'MAE': 197.48108836206896,
   'RMSE': 235.82696487522475,
   'R2': 0.7327310375948781,
   'MAPE': 0.785423540168574},
  'Low': {'MAE': 201.44669989224138,
   'RMSE': 241.18423638615775,
   'R2': 0.

In [17]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import ConvLSTM2D, Dropout, Flatten, Dense
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau



# ---------------- SEQUENCE CREATION ----------------
def create_seq(X, y, seq_len=25):
    X_seq, y_seq = [], []
    for i in range(seq_len, len(X)):
        X_seq.append(X[i-seq_len:i])
        y_seq.append(y[i])
    return np.array(X_seq), np.array(y_seq)



# ---------------- METRICS ----------------
def metrics(y_true, y_pred, label):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true,  y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    print(f"\n===== {label} =====")
    print("MAE :",  round(mae, 3))
    print("RMSE:",  round(rmse, 3))
    print("R2  :",  round(r2, 3))
    print("MAPE:", round(mape, 2), "%")

    return mae, rmse, r2, mape



# ---------------- BUILD ConvLSTM MODEL ----------------
def build_convlstm(input_shape):

    model = Sequential([

        ConvLSTM2D(
            filters=64,
            kernel_size=(1, 2),
            return_sequences=True,
            activation='tanh',
            padding='same',
            input_shape=input_shape
        ),
        Dropout(0.25),

        ConvLSTM2D(
            filters=32,
            kernel_size=(1, 2),
            return_sequences=False,
            activation='tanh',
            padding='same'
        ),
        Dropout(0.25),

        Flatten(),

        Dense(64, activation='relu'),
        Dense(2)
    ])

    model.compile(optimizer='adam', loss='mse')
    return model



# ---------------- MAIN FUNCTION ----------------
def run_ConvLSTM(df, name):

    print(f"\n================ ConvLSTM for {name} ================")

    # Features and target
    X = df[['Open', 'High', 'Low', 'Close']].values
    y = df[['High', 'Low']].values

    seq_len = 25
    train_size = max(int(len(df) * 0.85), seq_len + 3)
    train_size = min(train_size, len(df) - 2)

    X_train_raw = X[:train_size]
    X_test_raw  = X[train_size:]

    y_train_raw = y[:train_size]
    y_test_raw  = y[train_size:]

    # Scaling
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_train_scaled = scaler_X.fit_transform(X_train_raw)
    X_test_scaled  = scaler_X.transform(X_test_raw)

    y_train_scaled = scaler_y.fit_transform(y_train_raw)
    y_test_scaled  = scaler_y.transform(y_test_raw)

    # Merge for sequence creation
    X_scaled = np.vstack([X_train_scaled, X_test_scaled])
    y_scaled = np.vstack([y_train_scaled, y_test_scaled])

    # Create sequences
    X_seq, y_seq = create_seq(X_scaled, y_scaled, seq_len)

    # Split into train/test
    split = train_size - seq_len
    X_train_seq, X_test_seq = X_seq[:split], X_seq[split:]
    y_train, y_test = y_seq[:split], y_seq[split:]

    # ConvLSTM requires 5D input: (batch, seq_len, spatial_x, spatial_y, channels)
    X_train = X_train_seq.reshape(-1, seq_len, 1, X_train_raw.shape[1], 1)
    X_test  = X_test_seq.reshape(-1, seq_len, 1, X_train_raw.shape[1], 1)

    # Adaptive batch size
    batch_size = 8 if len(X_train) > 50 else 2

    # Build model
    model = build_convlstm((seq_len, 1, X_train_raw.shape[1], 1))

    # Callbacks
    es = EarlyStopping(patience=6, restore_best_weights=True)
    lr = ReduceLROnPlateau(factor=0.5, patience=3)

    # Train model
    model.fit(
        X_train, y_train,
        epochs=50,
        batch_size=batch_size,
        validation_data=(X_test, y_test) if len(X_test) > 1 else None,
        callbacks=[es, lr],
        verbose=1
    )

    # Predict
    pred_scaled = model.predict(X_test)
    pred = scaler_y.inverse_transform(pred_scaled)
    y_true = scaler_y.inverse_transform(y_test)

    pred_high, pred_low = pred[:, 0], pred[:, 1]
    true_high, true_low = y_true[:, 0], y_true[:, 1]

    m1 = metrics(true_high, pred_high, f"ConvLSTM {name} High")
    m2 = metrics(true_low,  pred_low,  f"ConvLSTM {name} Low")

    return (m1, m2)



# ================= RUN ConvLSTM ON ALL DATASETS ================= #
print("\n\n====== RUNNING ConvLSTM ON ALL DATASETS ======\n")

results_convlstm = {}

for df_regime, name in zip([df1, df2, df3, df4], ["DF1", "DF2", "DF3", "DF4"]):
    print(f"\n\n----- Training ConvLSTM for {name} -----")
    try:
        res = run_ConvLSTM(df_regime, name)
        results_convlstm[name] = {
            "High": {
                "MAE":  res[0][0],
                "RMSE": res[0][1],
                "R2":   res[0][2],
                "MAPE": res[0][3]
            },
            "Low": {
                "MAE":  res[1][0],
                "RMSE": res[1][1],
                "R2":   res[1][2],
                "MAPE": res[1][3]
            }
        }
    except Exception as e:
        print(f"Error in {name}: {e}")

print("\n\n============= FINAL ConvLSTM RESULTS =============\n")
results_convlstm




====== RUNNING ConvLSTM ON ALL DATASETS ======



----- Training ConvLSTM for DF1 -----

================ ConvLSTM for DF1 ================
Epoch 1/50


d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


83/83 ━━━━━━━━━━━━━━━━━━━━ 6s 28ms/step - loss: 0.1229 - val_loss: 0.0346 - learning_rate: 0.0010
Epoch 2/50
83/83 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - loss: 0.0513 - val_loss: 0.0195 - learning_rate: 0.0010
Epoch 3/50
83/83 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - loss: 0.0383 - val_loss: 0.0118 - learning_rate: 0.0010
Epoch 4/50
83/83 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - loss: 0.0351 - val_loss: 0.0311 - learning_rate: 0.0010
Epoch 5/50
83/83 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - loss: 0.0391 - val_loss: 0.0126 - learning_rate: 0.0010
Epoch 6/50
83/83 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - loss: 0.0322 - val_loss: 0.0067 - learning_rate: 0.0010
Epoch 7/50
83/83 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - loss: 0.0291 - val_loss: 0.0148 - learning_rate: 0.0010
Epoch 8/50
83/83 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - loss: 0.0278 - val_loss: 0.0104 - learning_rate: 0.0010
Epoch 9/50
83/83 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - loss: 0.0260 - val_loss: 0.0057 - learning_rate: 0.0010
Epoch 10/50
83/83 ━━━━━━━━━━━━

d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


230/230 ━━━━━━━━━━━━━━━━━━━━ 8s 24ms/step - loss: 0.0465 - val_loss: 0.0160 - learning_rate: 0.0010
Epoch 2/50
230/230 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - loss: 0.0141 - val_loss: 0.0339 - learning_rate: 0.0010
Epoch 3/50
230/230 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - loss: 0.0129 - val_loss: 0.0101 - learning_rate: 0.0010
Epoch 4/50
230/230 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - loss: 0.0099 - val_loss: 0.0095 - learning_rate: 0.0010
Epoch 5/50
230/230 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - loss: 0.0094 - val_loss: 0.0131 - learning_rate: 0.0010
Epoch 6/50
230/230 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - loss: 0.0076 - val_loss: 0.0081 - learning_rate: 0.0010
Epoch 7/50
230/230 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - loss: 0.0085 - val_loss: 0.0069 - learning_rate: 0.0010
Epoch 8/50
230/230 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - loss: 0.0085 - val_loss: 0.0064 - learning_rate: 0.0010
Epoch 9/50
230/230 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - loss: 0.0076 - val_loss: 0.0118 - learning_rate: 0.0010
Epoch 10/50


d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


50/50 ━━━━━━━━━━━━━━━━━━━━ 5s 32ms/step - loss: 0.1919 - val_loss: 0.0350 - learning_rate: 0.0010
Epoch 2/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - loss: 0.0419 - val_loss: 0.0231 - learning_rate: 0.0010
Epoch 3/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - loss: 0.0326 - val_loss: 0.0244 - learning_rate: 0.0010
Epoch 4/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - loss: 0.0256 - val_loss: 0.0254 - learning_rate: 0.0010
Epoch 5/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - loss: 0.0259 - val_loss: 0.0154 - learning_rate: 0.0010
Epoch 6/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - loss: 0.0199 - val_loss: 0.0151 - learning_rate: 0.0010
Epoch 7/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - loss: 0.0224 - val_loss: 0.0317 - learning_rate: 0.0010
Epoch 8/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - loss: 0.0212 - val_loss: 0.0184 - learning_rate: 0.0010
Epoch 9/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - loss: 0.0166 - val_loss: 0.0158 - learning_rate: 0.0010
Epoch 10/50
50/50 ━━━━━━━━━━━━

d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


100/100 ━━━━━━━━━━━━━━━━━━━━ 6s 27ms/step - loss: 0.0723 - val_loss: 0.0135 - learning_rate: 0.0010
Epoch 2/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - loss: 0.0278 - val_loss: 0.0254 - learning_rate: 0.0010
Epoch 3/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - loss: 0.0227 - val_loss: 0.0282 - learning_rate: 0.0010
Epoch 4/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - loss: 0.0191 - val_loss: 0.0225 - learning_rate: 0.0010
Epoch 5/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - loss: 0.0183 - val_loss: 0.0137 - learning_rate: 5.0000e-04
Epoch 6/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - loss: 0.0152 - val_loss: 0.0171 - learning_rate: 5.0000e-04
Epoch 7/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - loss: 0.0154 - val_loss: 0.0129 - learning_rate: 5.0000e-04
Epoch 8/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - loss: 0.0172 - val_loss: 0.0163 - learning_rate: 5.0000e-04
Epoch 9/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - loss: 0.0142 - val_loss: 0.0163 - learning_rate: 5.0

{'DF1': {'High': {'MAE': 51.80994252689549,
   'RMSE': 62.14051708957608,
   'R2': 0.957685847840025,
   'MAPE': 0.883989823064606},
  'Low': {'MAE': 51.618580302254095,
   'RMSE': 64.8726439439643,
   'R2': 0.9496176156186635,
   'MAPE': 0.8833183580465278}},
 'DF2': {'High': {'MAE': 73.22765482522796,
   'RMSE': 103.01766715877902,
   'R2': 0.960610066681788,
   'MAPE': 0.6495245089317568},
  'Low': {'MAE': 85.54095032294833,
   'RMSE': 111.16628865681943,
   'R2': 0.9560167138296616,
   'MAPE': 0.7688763694311308}},
 'DF3': {'High': {'MAE': 190.03677083333332,
   'RMSE': 242.5177921625151,
   'R2': 0.6508550163193962,
   'MAPE': 1.0749327943153377},
  'Low': {'MAE': 236.69229166666668,
   'RMSE': 310.97624088704947,
   'R2': 0.4844372728000583,
   'MAPE': 1.3612348375539858}},
 'DF4': {'High': {'MAE': 141.14692887931034,
   'RMSE': 181.48214237031752,
   'R2': 0.8417187762890957,
   'MAPE': 0.561953930408192},
  'Low': {'MAE': 150.9049838362069,
   'RMSE': 187.8053847656838,
   'R2'

In [18]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from catboost import CatBoostRegressor


# ---------------- SEQUENCE CREATION ----------------
def create_seq(X, y, seq_len=25):
    X_seq, y_seq = [], []
    for i in range(seq_len, len(X)):
        X_seq.append(X[i-seq_len:i].flatten())  # Flatten for CatBoost
        y_seq.append(y[i])
    return np.array(X_seq), np.array(y_seq)


# ---------------- METRICS ----------------
def metrics(y_true, y_pred, label):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true,  y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    print(f"\n===== {label} =====")
    print("MAE :",  round(mae, 3))
    print("RMSE:",  round(rmse, 3))
    print("R2  :",  round(r2, 3))
    print("MAPE:", round(mape, 2), "%")

    return mae, rmse, r2, mape


# ---------------- BUILD & RUN CATBOOST MODEL ----------------
def run_CatBoost(df, name):

    print(f"\n================ CatBoost for {name} ================")

    # Features and target
    X = df[['Open', 'High', 'Low', 'Close']].values
    y = df[['High', 'Low']].values

    seq_len = 25
    train_size = max(int(len(df) * 0.85), seq_len + 3)
    train_size = min(train_size, len(df) - 2)

    X_train_raw = X[:train_size]
    X_test_raw  = X[train_size:]

    y_train_raw = y[:train_size]
    y_test_raw  = y[train_size:]

    # Scaling
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_train_scaled = scaler_X.fit_transform(X_train_raw)
    X_test_scaled  = scaler_X.transform(X_test_raw)

    y_train_scaled = scaler_y.fit_transform(y_train_raw)
    y_test_scaled  = scaler_y.transform(y_test_raw)

    # Merge for sequence creation
    X_scaled = np.vstack([X_train_scaled, X_test_scaled])
    y_scaled = np.vstack([y_train_scaled, y_test_scaled])

    # Create sequences
    X_seq, y_seq = create_seq(X_scaled, y_scaled, seq_len)

    # Split into train/test
    split = train_size - seq_len
    X_train, X_test = X_seq[:split], X_seq[split:]
    y_train, y_test = y_seq[:split], y_seq[split:]

    # ---------------- CatBoost Model ----------------
    model = CatBoostRegressor(
        iterations=1000,
        learning_rate=0.05,
        depth=8,
        loss_function='MultiRMSE',
        verbose=False
    )

    # Train
    model.fit(X_train, y_train)

    # Predict
    pred_scaled = model.predict(X_test)
    pred = scaler_y.inverse_transform(pred_scaled)
    y_true = scaler_y.inverse_transform(y_test)

    pred_high, pred_low = pred[:, 0], pred[:, 1]
    true_high, true_low = y_true[:, 0], y_true[:, 1]

    m1 = metrics(true_high, pred_high, f"CatBoost {name} High")
    m2 = metrics(true_low,  pred_low,  f"CatBoost {name} Low")

    return (m1, m2)



# ================= RUN CATBOOST ON ALL DATASETS ================= #

print("\n\n====== RUNNING CatBoost ON ALL DATASETS ======\n")

results_catboost = {}

# df1, df2, df3, df4 must already be created before this
for df_regime, name in zip([df1, df2, df3, df4], ["DF1", "DF2", "DF3", "DF4"]):
    print(f"\n\n----- Training CatBoost for {name} -----")
    try:
        res = run_CatBoost(df_regime, name)
        results_catboost[name] = {
            "High": {
                "MAE":  res[0][0],
                "RMSE": res[0][1],
                "R2":   res[0][2],
                "MAPE": res[0][3]
            },
            "Low": {
                "MAE":  res[1][0],
                "RMSE": res[1][1],
                "R2":   res[1][2],
                "MAPE": res[1][3]
            }
        }
    except Exception as e:
        print(f"Error in {name}: {e}")

print("\n\n============= FINAL CatBoost RESULTS =============\n")
results_catboost




====== RUNNING CatBoost ON ALL DATASETS ======



----- Training CatBoost for DF1 -----

================ CatBoost for DF1 ================

===== CatBoost DF1 High =====
MAE : 71.718
RMSE: 89.734
R2  : 0.912
MAPE: 1.24 %

===== CatBoost DF1 Low =====
MAE : 89.474
RMSE: 119.897
R2  : 0.828
MAPE: 1.54 %


----- Training CatBoost for DF2 -----

================ CatBoost for DF2 ================

===== CatBoost DF2 High =====
MAE : 237.119
RMSE: 312.722
R2  : 0.637
MAPE: 2.03 %

===== CatBoost DF2 Low =====
MAE : 236.513
RMSE: 302.47
R2  : 0.674
MAPE: 2.06 %


----- Training CatBoost for DF3 -----

================ CatBoost for DF3 ================

===== CatBoost DF3 High =====
MAE : 422.315
RMSE: 514.357
R2  : -0.571
MAPE: 2.35 %

===== CatBoost DF3 Low =====
MAE : 386.631
RMSE: 468.019
R2  : -0.168
MAPE: 2.19 %


----- Training CatBoost for DF4 -----

================ CatBoost for DF4 ================

===== CatBoost DF4 High =====
MAE : 199.742
RMSE: 247.374
R2  : 0.706
MAPE: 0.79 %

{'DF1': {'High': {'MAE': 71.71756294900476,
   'RMSE': 89.73397972326023,
   'R2': 0.9117631945784153,
   'MAPE': 1.2433261814952423},
  'Low': {'MAE': 89.47437839561202,
   'RMSE': 119.89680107103072,
   'R2': 0.8279040253743241,
   'MAPE': 1.5403690804651544}},
 'DF2': {'High': {'MAE': 237.11920928813893,
   'RMSE': 312.722285981958,
   'R2': 0.6370226816747147,
   'MAPE': 2.034241228379973},
  'Low': {'MAE': 236.51333520497167,
   'RMSE': 302.47043761156806,
   'R2': 0.6743828056337203,
   'MAPE': 2.0638804284429777}},
 'DF3': {'High': {'MAE': 422.3145832759342,
   'RMSE': 514.3572777346384,
   'R2': -0.5705377267499774,
   'MAPE': 2.3514964801410607},
  'Low': {'MAE': 386.63051934432457,
   'RMSE': 468.01864970250256,
   'R2': -0.16775950780575455,
   'MAPE': 2.1897588601800586}},
 'DF4': {'High': {'MAE': 199.7423390791357,
   'RMSE': 247.3741962884752,
   'R2': 0.7059166771421954,
   'MAPE': 0.7901591615692523},
  'Low': {'MAE': 208.08467636115358,
   'RMSE': 262.32565023201016,
 

In [20]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.multioutput import MultiOutputRegressor

from xgboost import XGBRegressor


# ---------------- SEQUENCE CREATION ----------------
def create_seq(X, y, seq_len=25):
    X_seq, y_seq = [], []
    for i in range(seq_len, len(X)):
        X_seq.append(X[i-seq_len:i].flatten())  # flatten for XGBoost
        y_seq.append(y[i])
    return np.array(X_seq), np.array(y_seq)


# ---------------- METRICS ----------------
def metrics(y_true, y_pred, label):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true,  y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    print(f"\n===== {label} =====")
    print("MAE :",  round(mae, 3))
    print("RMSE:",  round(rmse, 3))
    print("R2  :",  round(r2, 3))
    print("MAPE:", round(mape, 2), "%")

    return mae, rmse, r2, mape


# ---------------- BUILD & RUN XGBOOST MODEL ----------------
def run_XGBoost(df, name):

    print(f"\n================ XGBoost for {name} ================")

    # Features and target
    X = df[['Open', 'High', 'Low', 'Close']].values
    y = df[['High', 'Low']].values   # Multi-output prediction

    seq_len = 25
    train_size = max(int(len(df) * 0.85), seq_len + 3)
    train_size = min(train_size, len(df) - 2)

    X_train_raw = X[:train_size]
    X_test_raw  = X[train_size:]

    y_train_raw = y[:train_size]
    y_test_raw  = y[train_size:]

    # Scaling
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_train_scaled = scaler_X.fit_transform(X_train_raw)
    X_test_scaled  = scaler_X.transform(X_test_raw)

    y_train_scaled = scaler_y.fit_transform(y_train_raw)
    y_test_scaled  = scaler_y.transform(y_test_raw)

    X_scaled = np.vstack([X_train_scaled, X_test_scaled])
    y_scaled = np.vstack([y_train_scaled, y_test_scaled])

    # Sequence creation
    X_seq, y_seq = create_seq(X_scaled, y_scaled, seq_len)

    split = train_size - seq_len
    X_train, X_test = X_seq[:split], X_seq[split:]
    y_train, y_test = y_seq[:split], y_seq[split:]


    # ---------------- XGBoost Model ----------------
    model = MultiOutputRegressor(
        XGBRegressor(
            n_estimators=600,
            learning_rate=0.05,
            max_depth=8,
            subsample=0.9,
            colsample_bytree=0.9,
            objective='reg:squarederror',
            tree_method="hist",
            eval_metric="rmse"
        )
    )

    # Train
    model.fit(X_train, y_train)

    # Predict
    pred_scaled = model.predict(X_test)
    pred = scaler_y.inverse_transform(pred_scaled)
    y_true = scaler_y.inverse_transform(y_test)

    pred_high, pred_low = pred[:, 0], pred[:, 1]
    true_high, true_low = y_true[:, 0], y_true[:, 1]

    m1 = metrics(true_high, pred_high, f"XGBoost {name} High")
    m2 = metrics(true_low, pred_low,  f"XGBoost {name} Low")

    return (m1, m2)




# ================= RUN XGBOOST ON ALL DATASETS ================= #

print("\n\n====== RUNNING XGBoost ON ALL DATASETS ======\n")

results_xgboost = {}

for df_regime, name in zip([df1, df2, df3, df4], ["DF1", "DF2", "DF3", "DF4"]):
    print(f"\n\n----- Training XGBoost for {name} -----")
    try:
        res = run_XGBoost(df_regime, name)
        results_xgboost[name] = {
            "High": {
                "MAE":  res[0][0],
                "RMSE": res[0][1],
                "R2":   res[0][2],
                "MAPE": res[0][3]
            },
            "Low": {
                "MAE":  res[1][0],
                "RMSE": res[1][1],
                "R2":   res[1][2],
                "MAPE": res[1][3]
            }
        }
    except Exception as e:
        print(f"Error in {name}: {e}")

print("\n\n============= FINAL XGBoost RESULTS =============\n")
results_xgboost




====== RUNNING XGBoost ON ALL DATASETS ======



----- Training XGBoost for DF1 -----

================ XGBoost for DF1 ================

===== XGBoost DF1 High =====
MAE : 47.628
RMSE: 60.996
R2  : 0.959
MAPE: 0.82 %

===== XGBoost DF1 Low =====
MAE : 166.736
RMSE: 255.574
R2  : 0.218
MAPE: 2.83 %


----- Training XGBoost for DF2 -----

================ XGBoost for DF2 ================

===== XGBoost DF2 High =====
MAE : 239.269
RMSE: 322.183
R2  : 0.615
MAPE: 2.04 %

===== XGBoost DF2 Low =====
MAE : 231.305
RMSE: 305.281
R2  : 0.668
MAPE: 2.01 %


----- Training XGBoost for DF3 -----

================ XGBoost for DF3 ================

===== XGBoost DF3 High =====
MAE : 420.659
RMSE: 510.73
R2  : -0.548
MAPE: 2.34 %

===== XGBoost DF3 Low =====
MAE : 392.45
RMSE: 474.113
R2  : -0.198
MAPE: 2.22 %


----- Training XGBoost for DF4 -----

================ XGBoost for DF4 ================

===== XGBoost DF4 High =====
MAE : 143.032
RMSE: 187.604
R2  : 0.831
MAPE: 0.56 %

===== XGBoost 

{'DF1': {'High': {'MAE': 47.62780161372951,
   'RMSE': 60.99575355691108,
   'R2': 0.9592305249118053,
   'MAPE': 0.8205621256864362},
  'Low': {'MAE': 166.735595703125,
   'RMSE': 255.57416180811708,
   'R2': 0.21803122677568276,
   'MAPE': 2.8328101214984645}},
 'DF2': {'High': {'MAE': 239.26900586531156,
   'RMSE': 322.1829857226689,
   'R2': 0.6147283726883828,
   'MAPE': 2.0443502500237276},
  'Low': {'MAE': 231.30504072473406,
   'RMSE': 305.281319821782,
   'R2': 0.6683027111120582,
   'MAPE': 2.0069374696583138}},
 'DF3': {'High': {'MAE': 420.65890625,
   'RMSE': 510.73014019083,
   'R2': -0.5484656334805822,
   'MAPE': 2.3432846472961995},
  'Low': {'MAE': 392.45002604166666,
   'RMSE': 474.1131314318644,
   'R2': -0.1983703675182631,
   'MAPE': 2.222286044447039}},
 'DF4': {'High': {'MAE': 143.03217941810345,
   'RMSE': 187.60444601752118,
   'R2': 0.830859403697948,
   'MAPE': 0.564450106810546},
  'Low': {'MAE': 178.6793911637931,
   'RMSE': 249.33983799275705,
   'R2': 0.7

In [4]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tcn import TCN
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


# ---------------- SEQUENCE CREATION ----------------
def create_seq(X, y, seq_len=25):
    X_seq, y_seq = [], []
    for i in range(seq_len, len(X)):
        X_seq.append(X[i-seq_len:i])
        y_seq.append(y[i])
    return np.array(X_seq), np.array(y_seq)


# ---------------- METRICS FUNCTION ----------------
def metrics(y_true, y_pred, label):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true,  y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    print(f"\n===== {label} =====")
    print("MAE :", round(mae, 4))
    print("RMSE:", round(rmse, 4))
    print("R2  :", round(r2, 4))
    print("MAPE:", round(mape, 4), "%")

    return mae, rmse, r2, mape


# ---------------- BUILD TCN MODEL ----------------
def build_tcn(input_shape):

    model = Sequential([
        TCN(
            nb_filters=64,
            kernel_size=3,
            dilations=[1, 2, 4, 8],
            dropout_rate=0.2,
            activation='relu',
            return_sequences=False,
            input_shape=input_shape
        ),
        Dense(64, activation='relu'),
        Dense(2)  # Predict High & Low
    ])

    model.compile(optimizer='adam', loss='mse')
    return model


# ---------------- MAIN FUNCTION ----------------
def run_TCN(df, name):

    print(f"\n================ TCN for {name} ================")

    # Features & target
    X = df[['Open', 'High', 'Low', 'Close']].values
    y = df[['High', 'Low']].values

    seq_len = 25
    train_size = max(int(len(df) * 0.85), seq_len + 3)
    train_size = min(train_size, len(df) - 2)

    # Split
    X_train_raw = X[:train_size]
    X_test_raw  = X[train_size:]

    y_train_raw = y[:train_size]
    y_test_raw  = y[train_size:]

    # Scaling
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_train_scaled = scaler_X.fit_transform(X_train_raw)
    X_test_scaled  = scaler_X.transform(X_test_raw)

    y_train_scaled = scaler_y.fit_transform(y_train_raw)
    y_test_scaled  = scaler_y.transform(y_test_raw)

    # Merge for sequences
    X_scaled = np.vstack([X_train_scaled, X_test_scaled])
    y_scaled = np.vstack([y_train_scaled, y_test_scaled])

    # Create sequences
    X_seq, y_seq = create_seq(X_scaled, y_scaled, seq_len)

    split = train_size - seq_len

    X_train, X_test = X_seq[:split], X_seq[split:]
    y_train, y_test = y_seq[:split], y_seq[split:]

    # Build TCN model
    model = build_tcn((seq_len, X.shape[1]))

    es = EarlyStopping(patience=6, restore_best_weights=True)
    lr = ReduceLROnPlateau(factor=0.5, patience=3)

    # Train model
    model.fit(
        X_train, y_train,
        epochs=50,
        batch_size=16,
        validation_data=(X_test, y_test) if len(X_test) > 1 else None,
        callbacks=[es, lr],
        verbose=1
    )

    # Predict
    pred_scaled = model.predict(X_test)
    pred = scaler_y.inverse_transform(pred_scaled)
    y_true = scaler_y.inverse_transform(y_test)

    pred_high, pred_low = pred[:, 0], pred[:, 1]
    true_high, true_low = y_true[:, 0], y_true[:, 1]

    m1 = metrics(true_high, pred_high, f"TCN {name} High")
    m2 = metrics(true_low,  pred_low,  f"TCN {name} Low")

    return (m1, m2)



# ------------ RUN TCN ON ALL DATASETS ------------
results_tcn = {}

for df_regime, name in zip([df1, df2, df3, df4], ["DF1", "DF2", "DF3", "DF4"]):
    print(f"\n----- Training TCN for {name} -----")
    try:
        res = run_TCN(df_regime, name)
        results_tcn[name] = {
            "High": {"MAE": res[0][0], "RMSE": res[0][1], "R2": res[0][2], "MAPE": res[0][3]},
            "Low":  {"MAE": res[1][0], "RMSE": res[1][1], "R2": res[1][2], "MAPE": res[1][3]}
        }
    except Exception as e:
        print(f"Error in {name}: {e}")


print("\n\n============= FINAL TCN RESULTS =============")
print(results_tcn)



----- Training TCN for DF1 -----

================ TCN for DF1 ================



d:\Anacondaa\envs\tfenv\lib\site-packages\tcn\tcn.py:268: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super(TCN, self).__init__(**kwargs)


Epoch 1/50
42/42 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 6.6827 - val_loss: 1.1005 - learning_rate: 0.0010
Epoch 2/50
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.8537 - val_loss: 0.1596 - learning_rate: 0.0010
Epoch 3/50
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.4150 - val_loss: 0.0960 - learning_rate: 0.0010
Epoch 4/50
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.3176 - val_loss: 0.1407 - learning_rate: 0.0010
Epoch 5/50
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.2538 - val_loss: 0.1494 - learning_rate: 0.0010
Epoch 6/50
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.2222 - val_loss: 0.0506 - learning_rate: 0.0010
Epoch 7/50
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.1863 - val_loss: 0.1132 - learning_rate: 0.0010
Epoch 8/50
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.1534 - val_loss: 0.2065 - learning_rate: 0.0010
Epoch 9/50
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.1532 - val_loss: 0.1455 - learning_rate: 0.0010
Epoch 10/50
42/42 ━━━━━━━━━

d:\Anacondaa\envs\tfenv\lib\site-packages\tcn\tcn.py:268: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super(TCN, self).__init__(**kwargs)


115/115 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - loss: 0.9822 - val_loss: 0.3018 - learning_rate: 0.0010
Epoch 2/50
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.1307 - val_loss: 0.2861 - learning_rate: 0.0010
Epoch 3/50
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0781 - val_loss: 0.2375 - learning_rate: 0.0010
Epoch 4/50
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0591 - val_loss: 0.1222 - learning_rate: 0.0010
Epoch 5/50
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0477 - val_loss: 0.0795 - learning_rate: 0.0010
Epoch 6/50
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0383 - val_loss: 0.1661 - learning_rate: 0.0010
Epoch 7/50
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0308 - val_loss: 0.0988 - learning_rate: 0.0010
Epoch 8/50
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0268 - val_loss: 0.1152 - learning_rate: 0.0010
Epoch 9/50
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0228 - val_loss: 0.1519 - learning_rate: 5.0000e-04
Epoch 10/50
115/1

d:\Anacondaa\envs\tfenv\lib\site-packages\tcn\tcn.py:268: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super(TCN, self).__init__(**kwargs)


25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - loss: 11.9280 - val_loss: 0.5361 - learning_rate: 0.0010
Epoch 2/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 1.3584 - val_loss: 0.0722 - learning_rate: 0.0010
Epoch 3/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.7582 - val_loss: 0.1139 - learning_rate: 0.0010
Epoch 4/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.5126 - val_loss: 0.0339 - learning_rate: 0.0010
Epoch 5/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.3995 - val_loss: 0.1435 - learning_rate: 0.0010
Epoch 6/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.3652 - val_loss: 0.0537 - learning_rate: 0.0010
Epoch 7/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.2990 - val_loss: 0.2106 - learning_rate: 0.0010
Epoch 8/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.2358 - val_loss: 0.1871 - learning_rate: 5.0000e-04
Epoch 9/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.2639 - val_loss: 0.1209 - learning_rate: 5.0000e-04
Epoch 10/50
25/25 ━━━━━━━━━━━

d:\Anacondaa\envs\tfenv\lib\site-packages\tcn\tcn.py:268: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super(TCN, self).__init__(**kwargs)


50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 5.4430 - val_loss: 0.0557 - learning_rate: 0.0010
Epoch 2/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.4405 - val_loss: 0.3952 - learning_rate: 0.0010
Epoch 3/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.2904 - val_loss: 0.1566 - learning_rate: 0.0010
Epoch 4/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.1923 - val_loss: 0.1015 - learning_rate: 0.0010
Epoch 5/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.1878 - val_loss: 0.1429 - learning_rate: 5.0000e-04
Epoch 6/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.1529 - val_loss: 0.0665 - learning_rate: 5.0000e-04
Epoch 7/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.1343 - val_loss: 0.1893 - learning_rate: 5.0000e-04
1/5 ━━━━━━━━━━━━━━━━━━━━ 0s 137ms/stepWARNING:tensorflow:6 out of the last 19 calls to <function TensorFlowTrainer.make_predict_function.<locals>.one_step_on_data_distributed at 0x00000276E8690D30> triggered tf.function retracing. Tracing is 

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


# ---------------- SEQUENCE CREATION ----------------
def create_seq(X, y, seq_len=25):
    X_seq, y_seq = [], []
    for i in range(seq_len, len(X)):
        X_seq.append(X[i-seq_len:i])
        y_seq.append(y[i])
    return np.array(X_seq), np.array(y_seq)


# ---------------- METRICS ----------------
def metrics(y_true, y_pred, label):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true,  y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    print(f"\n===== {label} =====")
    print("MAE :",  round(mae, 3))
    print("RMSE:",  round(rmse, 3))
    print("R2  :",  round(r2, 3))
    print("MAPE:", round(mape, 2), "%")

    return mae, rmse, r2, mape



# ---------------- LSTM MODEL ----------------
def build_lstm(input_shape):

    model = Sequential([
        LSTM(128, return_sequences=True, activation='tanh', input_shape=input_shape),
        Dropout(0.2),

        LSTM(64, return_sequences=False, activation='tanh'),
        Dropout(0.2),

        Dense(32, activation='relu'),
        Dense(2)     # Predicting High & Low
    ])

    model.compile(optimizer='adam', loss='mse')
    return model



# ---------------- MAIN LSTM RUN FUNCTION ----------------
def run_LSTM(df, name):

    print(f"\n================ LSTM for {name} ================")

    # Features and target
    X = df[['Open', 'High', 'Low', 'Close']].values
    y = df[['High', 'Low']].values

    seq_len = 25
    train_size = max(int(len(df) * 0.85), seq_len + 3)
    train_size = min(train_size, len(df) - 2)

    X_train_raw = X[:train_size]
    X_test_raw  = X[train_size:]

    y_train_raw = y[:train_size]
    y_test_raw  = y[train_size:]

    # Scaling
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_train_scaled = scaler_X.fit_transform(X_train_raw)
    X_test_scaled  = scaler_X.transform(X_test_raw)

    y_train_scaled = scaler_y.fit_transform(y_train_raw)
    y_test_scaled  = scaler_y.transform(y_test_raw)

    # Merge full scaled data for consistent sequencing
    X_scaled = np.vstack([X_train_scaled, X_test_scaled])
    y_scaled = np.vstack([y_train_scaled, y_test_scaled])

    # Create sequences
    X_seq, y_seq = create_seq(X_scaled, y_scaled, seq_len)

    # Split sequences into train/test
    split = train_size - seq_len
    X_train, X_test = X_seq[:split], X_seq[split:]
    y_train, y_test = y_seq[:split], y_seq[split:]

    # Build model
    model = build_lstm((seq_len, X_train_raw.shape[1]))

    # Callbacks
    es = EarlyStopping(patience=6, restore_best_weights=True)
    lr = ReduceLROnPlateau(factor=0.5, patience=3)

    # Train model
    model.fit(
        X_train, y_train,
        epochs=50,
        batch_size=16,
        validation_data=(X_test, y_test) if len(X_test) > 1 else None,
        callbacks=[es, lr],
        verbose=1
    )

    # Predict
    pred_scaled = model.predict(X_test)
    pred = scaler_y.inverse_transform(pred_scaled)
    y_true = scaler_y.inverse_transform(y_test)

    pred_high, pred_low = pred[:, 0], pred[:, 1]
    true_high, true_low = y_true[:, 0], y_true[:, 1]

    m1 = metrics(true_high, pred_high, f"LSTM {name} High")
    m2 = metrics(true_low,  pred_low,  f"LSTM {name} Low")

    return (m1, m2)



# ================= RUN LSTM ON ALL DATASETS ================= #
print("\n\n====== RUNNING LSTM ON ALL DATASETS ======\n")

results_lstm = {}

for df_regime, name in zip([df1, df2, df3, df4], ["DF1", "DF2", "DF3", "DF4"]):
    print(f"\n\n----- Training LSTM for {name} -----")
    try:
        res = run_LSTM(df_regime, name)
        results_lstm[name] = {
            "High": {
                "MAE":  res[0][0],
                "RMSE": res[0][1],
                "R2":   res[0][2],
                "MAPE": res[0][3]
            },
            "Low": {
                "MAE":  res[1][0],
                "RMSE": res[1][1],
                "R2":   res[1][2],
                "MAPE": res[1][3]
            }
        }
    except Exception as e:
        print(f"Error in {name}: {e}")

print("\n\n============= FINAL LSTM RESULTS =============\n")
results_lstm




====== RUNNING LSTM ON ALL DATASETS ======



----- Training LSTM for DF1 -----

================ LSTM for DF1 ================
Epoch 1/50


d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


42/42 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - loss: 0.1369 - val_loss: 0.0129 - learning_rate: 0.0010
Epoch 2/50
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0509 - val_loss: 0.0115 - learning_rate: 0.0010
Epoch 3/50
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0456 - val_loss: 0.0634 - learning_rate: 0.0010
Epoch 4/50
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0410 - val_loss: 0.0116 - learning_rate: 0.0010
Epoch 5/50
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0387 - val_loss: 0.0147 - learning_rate: 0.0010
Epoch 6/50
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0391 - val_loss: 0.0084 - learning_rate: 5.0000e-04
Epoch 7/50
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0355 - val_loss: 0.0227 - learning_rate: 5.0000e-04
Epoch 8/50
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0327 - val_loss: 0.0159 - learning_rate: 5.0000e-04
Epoch 9/50
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0301 - val_loss: 0.0109 - learning_rate: 5.0000e-04
Epoch 10/50
42

d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


115/115 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - loss: 0.0730 - val_loss: 0.0442 - learning_rate: 0.0010
Epoch 2/50
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0160 - val_loss: 0.0143 - learning_rate: 0.0010
Epoch 3/50
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0164 - val_loss: 0.0172 - learning_rate: 0.0010
Epoch 4/50
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0134 - val_loss: 0.0147 - learning_rate: 0.0010
Epoch 5/50
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0123 - val_loss: 0.0165 - learning_rate: 0.0010
Epoch 6/50
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0110 - val_loss: 0.0104 - learning_rate: 5.0000e-04
Epoch 7/50
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0109 - val_loss: 0.0112 - learning_rate: 5.0000e-04
Epoch 8/50
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0111 - val_loss: 0.0128 - learning_rate: 5.0000e-04
Epoch 9/50
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0092 - val_loss: 0.0109 - learning_rate: 5.0000e-04
Epo

d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - loss: 0.2544 - val_loss: 0.6943 - learning_rate: 0.0010
Epoch 2/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0579 - val_loss: 0.3259 - learning_rate: 0.0010
Epoch 3/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0430 - val_loss: 0.1026 - learning_rate: 0.0010
Epoch 4/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0422 - val_loss: 0.0671 - learning_rate: 0.0010
Epoch 5/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0318 - val_loss: 0.0449 - learning_rate: 0.0010
Epoch 6/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0259 - val_loss: 0.0197 - learning_rate: 0.0010
Epoch 7/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0282 - val_loss: 0.0247 - learning_rate: 0.0010
Epoch 8/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0294 - val_loss: 0.0299 - learning_rate: 0.0010
Epoch 9/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0238 - val_loss: 0.0263 - learning_rate: 0.0010
Epoch 10/50
25/25 ━━━━━━━━━━━━

d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - loss: 0.1820 - val_loss: 0.0994 - learning_rate: 0.0010
Epoch 2/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0315 - val_loss: 0.0266 - learning_rate: 0.0010
Epoch 3/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0286 - val_loss: 0.0152 - learning_rate: 0.0010
Epoch 4/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0237 - val_loss: 0.0186 - learning_rate: 0.0010
Epoch 5/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0243 - val_loss: 0.0417 - learning_rate: 0.0010
Epoch 6/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0215 - val_loss: 0.0206 - learning_rate: 0.0010
Epoch 7/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0193 - val_loss: 0.0130 - learning_rate: 5.0000e-04
Epoch 8/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0183 - val_loss: 0.0083 - learning_rate: 5.0000e-04
Epoch 9/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0178 - val_loss: 0.0082 - learning_rate: 5.0000e-04
Epoch 10/50
50/50 

{'DF1': {'High': {'MAE': 51.057609182889344,
   'RMSE': 63.291505396126134,
   'R2': 0.9561038158750412,
   'MAPE': 0.8670280882713032},
  'Low': {'MAE': 69.23705654456967,
   'RMSE': 85.4606949254097,
   'R2': 0.9125643868011962,
   'MAPE': 1.1843663818113253}},
 'DF2': {'High': {'MAE': 118.74965568009118,
   'RMSE': 153.80177205343665,
   'R2': 0.9122020244313708,
   'MAPE': 1.0506720572784638},
  'Low': {'MAE': 130.1632076367781,
   'RMSE': 165.18485864186673,
   'R2': 0.9028859798398041,
   'MAPE': 1.1711355481477264}},
 'DF3': {'High': {'MAE': 238.108125,
   'RMSE': 291.6876008941779,
   'R2': 0.4949265590551789,
   'MAPE': 1.335818792368153},
  'Low': {'MAE': 243.016640625,
   'RMSE': 331.6086937943852,
   'R2': 0.41375532722324093,
   'MAPE': 1.3997343351184952}},
 'DF4': {'High': {'MAE': 159.26225754310346,
   'RMSE': 198.41715941682835,
   'R2': 0.8108004669548912,
   'MAPE': 0.6333311001049283},
  'Low': {'MAE': 164.0527209051724,
   'RMSE': 202.14165192962298,
   'R2': 0.819

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Bidirectional, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


# ---------------- SEQUENCE CREATION ----------------
def create_seq(X, y, seq_len=25):
    X_seq, y_seq = [], []
    for i in range(seq_len, len(X)):
        X_seq.append(X[i-seq_len:i])
        y_seq.append(y[i])
    return np.array(X_seq), np.array(y_seq)


# ---------------- METRICS ----------------
def metrics(y_true, y_pred, label):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true,  y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    print(f"\n===== {label} =====")
    print("MAE :", round(mae, 3))
    print("RMSE:", round(rmse, 3))
    print("R2  :", round(r2, 3))
    print("MAPE:", round(mape, 2), "%")

    return mae, rmse, r2, mape



# ---------------- BiLSTM MODEL ----------------
def build_bilstm(input_shape):

    model = Sequential([

        Bidirectional(LSTM(128, return_sequences=True, activation='tanh'),
                      input_shape=input_shape),
        Dropout(0.2),

        Bidirectional(LSTM(64, return_sequences=False, activation='tanh')),
        Dropout(0.2),

        Dense(32, activation='relu'),
        Dense(2)  # Predicting High & Low
    ])

    model.compile(optimizer='adam', loss='mse')
    return model



# ---------------- MAIN BiLSTM RUN FUNCTION ----------------
def run_BiLSTM(df, name):

    print(f"\n================ BiLSTM for {name} ================")

    # Features and target
    X = df[['Open', 'High', 'Low', 'Close']].values
    y = df[['High', 'Low']].values

    seq_len = 25
    train_size = max(int(len(df) * 0.85), seq_len + 3)
    train_size = min(train_size, len(df) - 2)

    X_train_raw = X[:train_size]
    X_test_raw  = X[train_size:]

    y_train_raw = y[:train_size]
    y_test_raw  = y[train_size:]

    # Scaling
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_train_scaled = scaler_X.fit_transform(X_train_raw)
    X_test_scaled  = scaler_X.transform(X_test_raw)

    y_train_scaled = scaler_y.fit_transform(y_train_raw)
    y_test_scaled  = scaler_y.transform(y_test_raw)

    # Merge full scaled data for consistent sequencing
    X_scaled = np.vstack([X_train_scaled, X_test_scaled])
    y_scaled = np.vstack([y_train_scaled, y_test_scaled])

    # Create sequences
    X_seq, y_seq = create_seq(X_scaled, y_scaled, seq_len)

    # Split sequences
    split = train_size - seq_len
    X_train, X_test = X_seq[:split], X_seq[split:]
    y_train, y_test = y_seq[:split], y_seq[split:]

    # Build model
    model = build_bilstm((seq_len, X_train_raw.shape[1]))

    # Callbacks
    es = EarlyStopping(patience=6, restore_best_weights=True)
    lr = ReduceLROnPlateau(factor=0.5, patience=3)

    # Train
    model.fit(
        X_train, y_train,
        epochs=50,
        batch_size=16,
        validation_data=(X_test, y_test) if len(X_test) > 1 else None,
        callbacks=[es, lr],
        verbose=1
    )

    # Predict
    pred_scaled = model.predict(X_test)
    pred = scaler_y.inverse_transform(pred_scaled)
    y_true = scaler_y.inverse_transform(y_test)

    pred_high, pred_low = pred[:, 0], pred[:, 1]
    true_high, true_low = y_true[:, 0], y_true[:, 1]

    m1 = metrics(true_high, pred_high, f"BiLSTM {name} High")
    m2 = metrics(true_low,  pred_low,  f"BiLSTM {name} Low")

    return (m1, m2)



# ================= RUN BiLSTM ON ALL DATASETS ================= #
print("\n\n====== RUNNING BiLSTM ON ALL DATASETS ======\n")

results_bilstm = {}

for df_regime, name in zip([df1, df2, df3, df4], ["DF1", "DF2", "DF3", "DF4"]):
    print(f"\n\n----- Training BiLSTM for {name} -----")
    try:
        res = run_BiLSTM(df_regime, name)
        results_bilstm[name] = {
            "High": {
                "MAE":  res[0][0],
                "RMSE": res[0][1],
                "R2":   res[0][2],
                "MAPE": res[0][3]
            },
            "Low": {
                "MAE":  res[1][0],
                "RMSE": res[1][1],
                "R2":   res[1][2],
                "MAPE": res[1][3]
            }
        }
    except Exception as e:
        print(f"Error in {name}: {e}")

print("\n\n============= FINAL BiLSTM RESULTS =============\n")
results_bilstm




====== RUNNING BiLSTM ON ALL DATASETS ======



----- Training BiLSTM for DF1 -----

================ BiLSTM for DF1 ================
Epoch 1/50


d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


42/42 ━━━━━━━━━━━━━━━━━━━━ 4s 23ms/step - loss: 0.1568 - val_loss: 0.0320 - learning_rate: 0.0010
Epoch 2/50
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0550 - val_loss: 0.0174 - learning_rate: 0.0010
Epoch 3/50
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0357 - val_loss: 0.0199 - learning_rate: 0.0010
Epoch 4/50
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0403 - val_loss: 0.0096 - learning_rate: 0.0010
Epoch 5/50
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.0324 - val_loss: 0.0851 - learning_rate: 0.0010
Epoch 6/50
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.0328 - val_loss: 0.0140 - learning_rate: 0.0010
Epoch 7/50
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0301 - val_loss: 0.0480 - learning_rate: 0.0010
Epoch 8/50
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0264 - val_loss: 0.0068 - learning_rate: 5.0000e-04
Epoch 9/50
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0236 - val_loss: 0.0094 - learning_rate: 5.0000e-04
Epoch 10/50
42/42 ━━━━

d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


115/115 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step - loss: 0.0454 - val_loss: 0.0849 - learning_rate: 0.0010
Epoch 2/50
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0160 - val_loss: 0.0323 - learning_rate: 0.0010
Epoch 3/50
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0146 - val_loss: 0.0269 - learning_rate: 0.0010
Epoch 4/50
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0115 - val_loss: 0.0349 - learning_rate: 0.0010
Epoch 5/50
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0099 - val_loss: 0.0162 - learning_rate: 0.0010
Epoch 6/50
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0117 - val_loss: 0.0414 - learning_rate: 0.0010
Epoch 7/50
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0108 - val_loss: 0.0585 - learning_rate: 0.0010
Epoch 8/50
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0087 - val_loss: 0.0155 - learning_rate: 0.0010
Epoch 9/50
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0079 - val_loss: 0.0121 - learning_rate: 0.0010
Epoch 10/50


d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


25/25 ━━━━━━━━━━━━━━━━━━━━ 4s 28ms/step - loss: 0.1694 - val_loss: 0.0679 - learning_rate: 0.0010
Epoch 2/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0506 - val_loss: 0.0784 - learning_rate: 0.0010
Epoch 3/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0375 - val_loss: 0.0285 - learning_rate: 0.0010
Epoch 4/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0298 - val_loss: 0.0409 - learning_rate: 0.0010
Epoch 5/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0266 - val_loss: 0.0448 - learning_rate: 0.0010
Epoch 6/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0229 - val_loss: 0.0214 - learning_rate: 0.0010
Epoch 7/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0208 - val_loss: 0.0262 - learning_rate: 0.0010
Epoch 8/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0210 - val_loss: 0.0188 - learning_rate: 0.0010
Epoch 9/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0191 - val_loss: 0.0345 - learning_rate: 0.0010
Epoch 10/50
25/25 ━━━━━━━━━━━━

d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


50/50 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - loss: 0.0910 - val_loss: 0.0124 - learning_rate: 0.0010
Epoch 2/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.0267 - val_loss: 0.0152 - learning_rate: 0.0010
Epoch 3/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0272 - val_loss: 0.0251 - learning_rate: 0.0010
Epoch 4/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0204 - val_loss: 0.0079 - learning_rate: 0.0010
Epoch 5/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0188 - val_loss: 0.0204 - learning_rate: 0.0010
Epoch 6/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0184 - val_loss: 0.0073 - learning_rate: 0.0010
Epoch 7/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0190 - val_loss: 0.0289 - learning_rate: 0.0010
Epoch 8/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0140 - val_loss: 0.0089 - learning_rate: 0.0010
Epoch 9/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0158 - val_loss: 0.0344 - learning_rate: 0.0010
Epoch 10/50
50/50 ━━━━━━━━━━━━

{'DF1': {'High': {'MAE': 40.55987849001025,
   'RMSE': 50.77614095392588,
   'R2': 0.9717476030458182,
   'MAPE': 0.6903299463209398},
  'Low': {'MAE': 47.03861023949795,
   'RMSE': 59.00264615123524,
   'R2': 0.958322803377767,
   'MAPE': 0.8048702323423154}},
 'DF2': {'High': {'MAE': 155.83974282864742,
   'RMSE': 191.82417024468472,
   'R2': 0.863425858219649,
   'MAPE': 1.3620022978356094},
  'Low': {'MAE': 157.0897220507219,
   'RMSE': 191.30264118382402,
   'R2': 0.8697483022487299,
   'MAPE': 1.401229866249622}},
 'DF3': {'High': {'MAE': 265.49864583333334,
   'RMSE': 320.55923867941516,
   'R2': 0.3899924770429104,
   'MAPE': 1.4873794992920029},
  'Low': {'MAE': 266.38265625,
   'RMSE': 332.67488366059024,
   'R2': 0.40997947351480835,
   'MAPE': 1.5257920508196612}},
 'DF4': {'High': {'MAE': 161.61580010775862,
   'RMSE': 202.86262357577698,
   'R2': 0.8022276017034202,
   'MAPE': 0.6441271690255952},
  'Low': {'MAE': 151.80777209051723,
   'RMSE': 186.44503615476813,
   'R2'

In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


# ---------------- SEQUENCE CREATION ----------------
def create_seq(X, y, seq_len=25):
    X_seq, y_seq = [], []
    for i in range(seq_len, len(X)):
        X_seq.append(X[i-seq_len:i])
        y_seq.append(y[i])
    return np.array(X_seq), np.array(y_seq)



# ---------------- METRICS ----------------
def metrics(y_true, y_pred, label):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true,  y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    print(f"\n===== {label} =====")
    print("MAE :", round(mae, 3))
    print("RMSE:", round(rmse, 3))
    print("R2  :", round(r2, 3))
    print("MAPE:", round(mape, 2), "%")

    return mae, rmse, r2, mape



# ---------------- BUILD Conv1D + LSTM MODEL ----------------
def build_conv_lstm_model(seq_len, num_features):

    model = Sequential([
        Conv1D(filters=64, kernel_size=3, activation='relu', padding='same',
               input_shape=(seq_len, num_features)),
        Dropout(0.2),

        Conv1D(filters=32, kernel_size=3, activation='relu', padding='same'),
        Dropout(0.2),

        LSTM(64, return_sequences=False),
        Dropout(0.2),

        Dense(32, activation='relu'),
        Dense(2)   # Output: High, Low
    ])

    model.compile(optimizer='adam', loss='mse')
    return model



# ---------------- MAIN FUNCTION ----------------
def run_ConvLSTM(df, name):

    print(f"\n================ Conv1D-LSTM for {name} ================")

    # Features and target
    X = df[['Open', 'High', 'Low', 'Close']].values
    y = df[['High', 'Low']].values

    seq_len = 25
    train_size = max(int(len(df) * 0.85), seq_len + 3)
    train_size = min(train_size, len(df) - 2)

    # Scale data
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_scaled = scaler_X.fit_transform(X)
    y_scaled = scaler_y.fit_transform(y)

    # Create sequences
    X_seq, y_seq = create_seq(X_scaled, y_scaled, seq_len)

    # Dataset split
    split = train_size - seq_len
    X_train, X_test = X_seq[:split], X_seq[split:]
    y_train, y_test = y_seq[:split], y_seq[split:]

    print("Training samples:", len(X_train), " | Testing samples:", len(X_test))

    # Build model
    model = build_conv_lstm_model(seq_len, X.shape[1])

    # Callbacks
    es = EarlyStopping(patience=8, restore_best_weights=True)
    lr = ReduceLROnPlateau(factor=0.5, patience=4)

    # Train model
    model.fit(
        X_train, y_train,
        epochs=50,
        batch_size=16,
        validation_data=(X_test, y_test),
        callbacks=[es, lr],
        verbose=1
    )

    # Predict
    pred_scaled = model.predict(X_test)
    pred = scaler_y.inverse_transform(pred_scaled)
    y_true = scaler_y.inverse_transform(y_test)

    pred_high, pred_low = pred[:, 0], pred[:, 1]
    true_high, true_low = y_true[:, 0], y_true[:, 1]

    m1 = metrics(true_high, pred_high, f"ConvLSTM {name} High")
    m2 = metrics(true_low,  pred_low,  f"ConvLSTM {name} Low")

    return (m1, m2)



# ================= RUN ConvLSTM ON ALL DATASETS ================= #
print("\n\n====== RUNNING Conv1D-LSTM ON ALL DATASETS ======\n")

results_convlstm = {}

for df_regime, name in zip([df1, df2, df3, df4], ["DF1", "DF2", "DF3", "DF4"]):
    print(f"\n\n----- Training Conv1D-LSTM for {name} -----")
    try:
        res = run_ConvLSTM(df_regime, name)
        results_convlstm[name] = {
            "High": {
                "MAE":  res[0][0],
                "RMSE": res[0][1],
                "R2":   res[0][2],
                "MAPE": res[0][3]
            },
            "Low": {
                "MAE":  res[1][0],
                "RMSE": res[1][1],
                "R2":   res[1][2],
                "MAPE": res[1][3]
            }
        }
    except Exception as e:
        print(f"Error in {name}: {e}")

print("\n\n============= FINAL Conv1D-LSTM RESULTS =============\n")
results_convlstm




====== RUNNING Conv1D-LSTM ON ALL DATASETS ======



----- Training Conv1D-LSTM for DF1 -----

================ Conv1D-LSTM for DF1 ================
Training samples: 661  | Testing samples: 122
Epoch 1/50


d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


42/42 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - loss: 0.2199 - val_loss: 0.0297 - learning_rate: 0.0010
Epoch 2/50
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0537 - val_loss: 0.0175 - learning_rate: 0.0010
Epoch 3/50
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0438 - val_loss: 0.0065 - learning_rate: 0.0010
Epoch 4/50
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0410 - val_loss: 0.0074 - learning_rate: 0.0010
Epoch 5/50
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0421 - val_loss: 0.0303 - learning_rate: 0.0010
Epoch 6/50
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0360 - val_loss: 0.0124 - learning_rate: 0.0010
Epoch 7/50
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0402 - val_loss: 0.0207 - learning_rate: 0.0010
Epoch 8/50
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0326 - val_loss: 0.0073 - learning_rate: 5.0000e-04
Epoch 9/50
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0270 - val_loss: 0.0094 - learning_rate: 5.0000e-04
Epoch 10/50
42/42 ━━━━━━━━━━━━

d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


115/115 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.0663 - val_loss: 0.0518 - learning_rate: 0.0010
Epoch 2/50
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0153 - val_loss: 0.0412 - learning_rate: 0.0010
Epoch 3/50
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0125 - val_loss: 0.0485 - learning_rate: 0.0010
Epoch 4/50
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0122 - val_loss: 0.0232 - learning_rate: 0.0010
Epoch 5/50
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0108 - val_loss: 0.0147 - learning_rate: 0.0010
Epoch 6/50
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0115 - val_loss: 0.0366 - learning_rate: 0.0010
Epoch 7/50
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0105 - val_loss: 0.0412 - learning_rate: 0.0010
Epoch 8/50
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0098 - val_loss: 0.0353 - learning_rate: 0.0010
Epoch 9/50
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0104 - val_loss: 0.1124 - learning_rate: 0.0010
Epoch 10/50
115/115 ━

d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - loss: 0.2850 - val_loss: 0.0467 - learning_rate: 0.0010
Epoch 2/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0486 - val_loss: 0.0507 - learning_rate: 0.0010
Epoch 3/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0395 - val_loss: 0.0307 - learning_rate: 0.0010
Epoch 4/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0321 - val_loss: 0.0190 - learning_rate: 0.0010
Epoch 5/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0237 - val_loss: 0.0207 - learning_rate: 0.0010
Epoch 6/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0234 - val_loss: 0.0473 - learning_rate: 0.0010
Epoch 7/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0220 - val_loss: 0.0142 - learning_rate: 0.0010
Epoch 8/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0223 - val_loss: 0.0192 - learning_rate: 0.0010
Epoch 9/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0218 - val_loss: 0.0153 - learning_rate: 0.0010
Epoch 10/50
25/25 ━━━━━━━━━━━━━━━━━━━━

d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


50/50 ━━━━━━━━━━━━━━━━━━━━ 3s 23ms/step - loss: 0.1128 - val_loss: 0.0173 - learning_rate: 0.0010
Epoch 2/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0269 - val_loss: 0.0107 - learning_rate: 0.0010
Epoch 3/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0243 - val_loss: 0.0210 - learning_rate: 0.0010
Epoch 4/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0255 - val_loss: 0.0182 - learning_rate: 0.0010
Epoch 5/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0214 - val_loss: 0.0069 - learning_rate: 0.0010
Epoch 6/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0242 - val_loss: 0.0129 - learning_rate: 0.0010
Epoch 7/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0202 - val_loss: 0.0112 - learning_rate: 0.0010
Epoch 8/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0191 - val_loss: 0.0060 - learning_rate: 0.0010
Epoch 9/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0220 - val_loss: 0.0100 - learning_rate: 0.0010
Epoch 10/50
50/50 ━━━━━━━━━━━━━━━━━━━━

{'DF1': {'High': {'MAE': 38.09459848872951,
   'RMSE': 49.729917312043376,
   'R2': 0.972899868889152,
   'MAPE': 0.6441352992029807},
  'Low': {'MAE': 38.951023789702866,
   'RMSE': 50.39245238789014,
   'R2': 0.9695990952614173,
   'MAPE': 0.6661903694564544}},
 'DF2': {'High': {'MAE': 220.57375866736322,
   'RMSE': 257.74494619163806,
   'R2': 0.7534289182394212,
   'MAPE': 1.9585024603916812},
  'Low': {'MAE': 228.53951367781156,
   'RMSE': 271.0128982858764,
   'R2': 0.7385904906159422,
   'MAPE': 2.0421160113703425}},
 'DF3': {'High': {'MAE': 159.57286458333334,
   'RMSE': 202.87647079075657,
   'R2': 0.7556670954774094,
   'MAPE': 0.9000515255465701},
  'Low': {'MAE': 192.149140625,
   'RMSE': 256.90975739681215,
   'R2': 0.6481250978185956,
   'MAPE': 1.1046694656040665}},
 'DF4': {'High': {'MAE': 135.29053071120688,
   'RMSE': 173.55646810028355,
   'R2': 0.8552417875381709,
   'MAPE': 0.5351255054207034},
  'Low': {'MAE': 174.32501346982758,
   'RMSE': 214.67167758412737,
   

In [5]:
import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis, jarque_bera
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import matplotlib.pyplot as plt


In [3]:
def get_stats(df, target="Close"):
    s = df[target].dropna()

    stats = {}

    # Basic statistics
    stats["Mean"] = s.mean()
    stats["Std"]  = s.std()
    stats["CV"]   = s.std() / s.mean()

    # Distribution shape
    stats["Skewness"] = skew(s)
    stats["Kurtosis"] = kurtosis(s, fisher=True)

    # Normality
    stats["JB_pvalue"] = jarque_bera(s)[1]

    # Stationarity
    stats["ADF_pvalue"] = adfuller(s)[1]

    try:
        stats["KPSS_pvalue"] = kpss(s, regression="c")[1]
    except:
        stats["KPSS_pvalue"] = np.nan

    return stats


In [6]:
results = pd.DataFrame({
    "FULL_DATA": get_stats(df),
    "DF1": get_stats(df1),
    "DF2": get_stats(df2),
    "DF3": get_stats(df3),
    "DF4": get_stats(df4)
})

results = results.round(4)
results


C:\Users\ASUS\AppData\Local\Temp\ipykernel_42328\995538300.py:22: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  stats["KPSS_pvalue"] = kpss(s, regression="c")[1]
C:\Users\ASUS\AppData\Local\Temp\ipykernel_42328\995538300.py:22: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  stats["KPSS_pvalue"] = kpss(s, regression="c")[1]
C:\Users\ASUS\AppData\Local\Temp\ipykernel_42328\995538300.py:22: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  stats["KPSS_pvalue"] = kpss(s, regression="c")[1]
C:\Users\ASUS\AppData\Local\Temp\ipykernel_42328\995538300.py:22: InterpolationWarning: The test statistic is outside of the range of p-values av

,FULL_DATA,DF1,DF2,DF3,DF4
Mean,10829.5796,4732.3498,8016.0592,13580.0928,20930.1571
Std,6250.1895,966.5794,2179.0326,2787.6388,3196.9176
CV,0.5771,0.2042,0.2718,0.2053,0.1527
Skewness,0.9211,-0.6917,0.1963,-0.0587,0.0325
Kurtosis,-0.2887,-0.3105,-1.2311,-1.1174,-1.5086
JB_pvalue,0.0000,0.0000,0.0000,0.0000,0.0000
ADF_pvalue,0.9973,0.7374,0.9575,0.9396,0.8994
KPSS_pvalue,0.0100,0.0100,0.0100,0.0100,0.0100
